# PV Fault Detection — final pipeline (simulated datasets), v3
Classes: **0 = normal, 1 = partial shading, 2 = dirt**.

**Rules used everywhere**
* Every score is on rows the model never saw. Split = 80 % train / 20 % test, stratified, seeds 0–4 (5 runs, like the paper).
* "Tuned" settings are chosen by 5-fold cross-validation on the **training rows only** (ANN: a validation split of the training rows).
* Extra features (Isc/G, physics ratios) are kept for a model only if they raise **its own training-data score**, never by looking at the test set.

**Part 1** shows every saved result instantly. **Part 2** re-runs the experiments: A) System 1 · C) System 2 zero-shot + few-shot · D) System 2 within-system (side result) · F) paper-style outputs · G) noise-robust training · E) summary. All data is simulated.
Part 2 takes about 1–1.5 h on Colab CPU with the ANN, about 35 min with `RUN_ANN = False`.

In [1]:
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip -q install xgboost
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/PV_Project/PV_final'   # <-- folder with the CSVs
else:
    DATA_DIR = os.environ.get('PV_DATA_DIR', '.')
os.chdir(DATA_DIR)
RUN_ANN = True
RUNS = int(os.environ.get('PV_RUNS', 5))
try:
    display
except NameError:
    display = print

Mounted at /content/drive


In [2]:
import warnings; warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

F4 = ['Voc/MaxVoc', 'Isc/MaxIsc', 'G/1000', 'AT/50']
IV = ['Vmp/MaxVoc', 'Imp/MaxIsc', 'FF']
load = lambda f: pd.read_csv(f)
S1 = {'realistic_hires_IV': load('S1_sim500_realistic_hires_IV.csv'), 'realistic_IV': load('S1_sim500_realistic_IV.csv'),
      'realistic_4f': load('S1_sim500_realistic_4features.csv')}
S2 = {'realistic_4f': load('S2_sim200_realistic_4features.csv'), 'realistic_IV': load('S2_sim200_realistic_IV.csv')}

def feats(df, variant='base'):
    """base: columns in the file (4 features, or 4 + I-V); iscg: + Isc/G; phys: + Isc/G and three I-V ratios."""
    x = df[[c for c in F4 + IV if c in df.columns]].copy()
    if variant in ('iscg', 'phys'):
        x['Isc_per_G'] = x['Isc/MaxIsc'] / x['G/1000']
    if variant == 'phys':
        x['Vmp_over_Voc'] = x['Vmp/MaxVoc'] / x['Voc/MaxVoc']   # shading moves the knee to lower voltage
        x['Imp_over_Isc'] = x['Imp/MaxIsc'] / x['Isc/MaxIsc']   # shading: MPP current falls below Isc
        x['Imp_per_G'] = x['Imp/MaxIsc'] / x['G/1000']          # dirt: less MPP current per unit of sunlight
    return x

def metrics(y, p):
    return dict(Accuracy=accuracy_score(y, p), Precision=precision_score(y, p, average='weighted', zero_division=0),
                Recall=recall_score(y, p, average='weighted', zero_division=0), F1=f1_score(y, p, average='weighted', zero_division=0))

def split(d, run):
    return train_test_split(d.index, test_size=0.2, stratify=d.Fault, random_state=run)

## Part 1 — All saved results (instant)

In [3]:
try:
    from IPython.display import Markdown
    display(Markdown(open('RESULTS_presentation.md').read()))
except Exception as e:
    print('Could not show RESULTS_presentation.md:', e)

# PV fault detection — results for the presentation

All results are on **simulated** data made by the calibrated physics simulator. Every score is accuracy on rows the model never saw (80 % train / 20 % test, 5 runs, like the paper). Tuned settings and feature choices use the training rows only. Weather (sunlight, temperature) is independent of the fault, as in reality.

**Feature sets.** *4 features*: the paper's Voc, Isc, G, T. *I-V*: + Vmp, Imp, fill factor (extra measurements from the I-V curve). *Isc/G*: current per unit of sunlight (physics feature). *Physics ratios*: Isc/G + Vmp/Voc + Imp/Isc + Imp/G.

## Part A — System 1

### A1. Step-by-step improvement (accuracy, "without → with Isc/G")

ANN (tuned) chooses Isc/G itself on validation rows, so it has one number per column. Paper ANN value is its own ANN (0.983).

| Model | Realistic, 4 features | Realistic + I-V | Realistic + I-V, high-res sensor | Paper (real data) |
|---|---|---|---|---|
| ANN (tuned) | 0.794 | 0.936 | 0.970 | 0.983 |
| XGBoost (tuned) | 0.716 → 0.810 | 0.874 → 0.942 | 0.884 → 0.972 | — |
| SVM (tuned) | 0.808 → 0.816 | 0.940 → 0.930 | 0.972 → 0.972 | 0.977 |
| SVM (paper) | 0.778 → 0.786 | 0.916 → 0.938 | 0.956 → 0.966 | 0.977 |
| RF (paper) | 0.652 → 0.806 | 0.840 → 0.940 | 0.880 → 0.976 | 0.957 |
| DT (paper) | 0.654 → 0.742 | 0.750 → 0.906 | 0.832 → 0.950 | 0.930 |
| LR (paper) | 0.822 → 0.830 | 0.952 → 0.948 | 0.974 → 0.974 | 0.787 |

### A2. Final System 1 models (realistic + I-V + high-res sensor)

The ANN was offered only Isc/G as an extra feature (not the three other ratios); see notes.

| Model | Features (chosen on training data) | Accuracy | Precision | Recall | F1 | Std | Paper accuracy (real) |
|---|---|---|---|---|---|---|---|
| ANN (tuned) | I-V, Isc/G chosen per run | 0.970 | 0.972 | 0.970 | 0.970 | 0.021 | 0.983 |
| XGBoost (tuned) | I-V + Isc/G | 0.972 | 0.973 | 0.972 | 0.972 | 0.011 | — |
| SVM (tuned) | I-V + physics ratios | 0.972 | 0.975 | 0.972 | 0.972 | 0.022 | 0.977 |
| RF (paper) | I-V + physics ratios | 0.974 | 0.976 | 0.974 | 0.974 | 0.015 | 0.957 |
| DT (paper) | I-V + physics ratios | 0.958 | 0.959 | 0.958 | 0.958 | 0.024 | 0.930 |
| LR (paper) | I-V + physics ratios | 0.976 | 0.978 | 0.976 | 0.976 | 0.018 | 0.787 |

### A3. Confusion matrices (run closest to the mean; rows/columns = normal, shading, dirt)

- **ANN (tuned)** (run 2): [[33, 0, 1], [0, 33, 0], [2, 0, 31]]
- **XGBoost (tuned)** (run 0): [[33, 0, 1], [0, 33, 0], [2, 0, 31]]
- **SVM (tuned)** (run 0): [[34, 0, 0], [0, 33, 0], [2, 1, 30]]
- **RF (paper)** (run 0): [[33, 0, 1], [0, 33, 0], [2, 0, 31]]
- **DT (paper)** (run 0): [[33, 0, 1], [0, 33, 0], [2, 1, 30]]
- **LR (paper)** (run 0): [[34, 0, 0], [0, 33, 0], [2, 0, 31]]

### A4. ANOVA (randomized block: model × run)

| Metric | F | p | partial η² |
|---|---|---|---|
| Accuracy | 2.65 | 0.054 | 0.399 |
| Precision | 3.07 | 0.032 | 0.434 |
| Recall | 2.65 | 0.054 | 0.399 |
| F1 | 2.65 | 0.054 | 0.398 |

p ≈ 0.05: the final models are statistically about equally good (the paper found strong differences, p ≈ 5×10⁻¹¹).

### A5. Sensitivity to measurement error on the test voltages and currents (as in the paper's Fig. 11)


| Model | ±0 % | ±1 % | ±5 % | ±10 % | ±15 % | ±20 % |
|---|---|---|---|---|---|---|
| ANN (tuned) | 0.970 | 0.934 | 0.719 | 0.613 | 0.552 | 0.513 |
| XGBoost (tuned) | 0.972 | 0.911 | 0.619 | 0.524 | 0.505 | 0.499 |
| SVM (tuned) | 0.972 | 0.968 | 0.867 | 0.719 | 0.643 | 0.583 |
| RF (paper) | 0.974 | 0.964 | 0.671 | 0.571 | 0.527 | 0.518 |
| DT (paper) | 0.958 | 0.934 | 0.658 | 0.565 | 0.517 | 0.510 |
| LR (paper) | 0.976 | 0.971 | 0.831 | 0.687 | 0.628 | 0.584 |
| Paper ANN (Fig. 11) | 0.983 | 0.933 | 0.667 | 0.533 | 0.450 | 0.417 |

### A6. After noise-robust training (training rows + copies with ±1 % and ±3 % error)

Not run for the ANN and DT.

| Model | ±0 % | ±1 % | ±5 % | ±10 % |
|---|---|---|---|---|
| LR (paper) | 0.968 | 0.967 | 0.898 | 0.745 |
| RF (paper) | 0.966 | 0.968 | 0.885 | 0.683 |
| SVM (tuned) | 0.932 | 0.933 | 0.785 | 0.419 |
| XGBoost (tuned) | 0.954 | 0.949 | 0.844 | 0.660 |

## Part B — System 2: zero-shot and few-shot

Models are trained on simulated System 1 and tested on simulated System 2. *Zero-shot*: no System 2 data in training. *Few-shot k*: k System 2 rows per class added to training (weighted 5×), tested on the remaining System 2 rows (5 draws).

### B1. System 2 accuracy ("without → with Isc/G")

Paper (trained on real System 1, tested on its real System 2): best SVM 0.627, RF 0.540, ANN 0.527, LR 0.513, DT 0.467.

| Model | 4 feat. zero-shot | 4 feat. k=5 | 4 feat. k=10 | I-V zero-shot | I-V k=5 | I-V k=10 |
|---|---|---|---|---|---|---|
| ANN (tuned) | 0.696 | 0.909 | 0.966 | 0.472 | 0.906 | 0.956 |
| XGBoost (tuned) | 0.800 → 0.805 | 0.907 → 0.995 | 0.954 → 0.996 | 0.300 → 0.345 | 0.923 → 0.895 | 0.960 → 0.952 |
| SVM (tuned) | 0.805 → 0.800 | 0.965 → 0.966 | 0.991 → 0.994 | 0.420 → 0.410 | 0.901 → 0.872 | 0.932 → 0.951 |
| SVM (paper) | 0.660 → 0.895 | 0.962 → 0.965 | 0.976 → 0.993 | 0.335 → 0.335 | 0.908 → 0.892 | 0.969 → 0.967 |
| RF (paper) | 0.785 → 0.805 | 0.936 → 0.973 | 0.976 → 0.994 | 0.365 → 0.525 | 0.912 → 0.933 | 0.942 → 0.959 |
| DT (paper) | 0.695 → 0.775 | 0.785 → 0.945 | 0.914 → 0.944 | 0.335 → 0.335 | 0.941 → 0.870 | 0.954 → 0.944 |
| LR (paper) | 0.800 → 0.800 | 0.965 → 0.969 | 0.979 → 0.973 | 0.420 → 0.420 | 0.945 → 0.937 | 0.973 → 0.972 |

### B2. Side result: trained and tested on simulated System 2 (80/20, 5 runs)


| Model | 4 features | I-V |
|---|---|---|
| ANN (tuned) | 1.000 | 0.975 |
| XGBoost (tuned) | 0.935 → 0.995 | 0.980 → 0.980 |
| SVM (tuned) | 0.990 → 0.990 | 0.985 → 0.985 |
| SVM (paper) | 0.985 → 1.000 | 0.985 → 0.980 |
| RF (paper) | 0.955 → 0.995 | 0.990 → 0.990 |
| DT (paper) | 0.910 → 0.995 | 0.975 → 0.980 |
| LR (paper) | 0.995 → 1.000 | 0.995 → 0.995 |

### B3. Our simulated System 2 vs the paper's real System 2

| | Paper (real) | Ours (simulated) |
|---|---|---|
| Hardware | 7 × 200 W modules in series, PHB1500-NS inverter | same layout and module ratings |
| Module health | 2 modules with a hotspot cell, 2 with bad cell connections | all modules healthy |
| Data | 60 rows (30 distinct readings, each recorded twice) | 200 distinct rows, balanced |
| Weather | each fault recorded in its own session (~14.5 °C or ~28 °C) | sunlight/temperature random over the paper's range, same for every class |
| Shading | chairs in front of the modules | shadow over 6–10 neighbouring module sections, calibrated to the real readings |
| Dirt | splashed dirt | light film + blotches incl. a few heavy ones, calibrated to the real readings |
| Measurements | inverter MPP converted to Voc/Isc | same (4 features); plus an I-V-tracer version |
| Match to real data | — | normal and dirt close; our shading voltage higher (~0.74 vs ~0.61) |

## Notes and limitations
- Simulated System 2 is healthy and idealised, so its numbers are not a like-for-like comparison with the paper's real System 2 (0.627).
- The ANN was tuned over learning rate and Isc/G only; the three extra physics ratios were offered to the other models but not to the ANN.
- XGBoost is the model most sensitive to measurement error (A5); noise-robust training reduces this at a small cost on clean data (A6).
- System 1 light dirt that changes the current by less than the sensor noise is physically undetectable; this sets the ~97 % ceiling.

## Part 2 — Re-run the experiments
### Models
* **paper**: hyperparameters exactly as in the paper's Table 3. **SVM / XGBoost (tuned)**: 5-fold CV on the training rows.
* **ANN (tuned)**: paper architecture (74-78-91), standardised inputs; learning rate (0.001 / 0.0062) and Isc/G chosen on a validation split of the training rows, epochs by early stopping on that split; then retrained on all training rows.

In [4]:
PAPER = {
    'SVM (paper)': lambda s: SVC(kernel='rbf', gamma=79.783, C=3.7442),
    'RF (paper)':  lambda s: RandomForestClassifier(1000, criterion='entropy', random_state=s, n_jobs=-1),
    'DT (paper)':  lambda s: DecisionTreeClassifier(criterion='entropy', min_samples_split=3, random_state=s),
    'LR (paper)':  lambda s: LogisticRegression(C=44.1871, class_weight='balanced', penalty='l1', solver='saga', max_iter=3000),
}
XGB_SPACE = {'n_estimators': [100, 200, 400, 600], 'max_depth': [2, 3, 4, 6], 'learning_rate': [0.03, 0.05, 0.1, 0.2],
             'subsample': [0.7, 0.85, 1.0], 'colsample_bytree': [0.6, 0.8, 1.0], 'min_child_weight': [1, 3, 5]}

def tune(name, X, y, s):
    cv = StratifiedKFold(5, shuffle=True, random_state=s)
    if name == 'SVM (tuned)':
        g = GridSearchCV(SVC(kernel='rbf'), {'gamma': np.logspace(-1, 3, 9), 'C': np.logspace(-1, 3, 9)}, cv=cv)
    else:
        g = RandomizedSearchCV(XGBClassifier(random_state=s, n_jobs=1), XGB_SPACE, n_iter=25, cv=cv, random_state=s)
    return g.fit(X, y)

def all_models(X, y, s):
    m = {n: f(s).fit(X, y) for n, f in PAPER.items()}
    for n in ['SVM (tuned)', 'XGBoost (tuned)']:
        m[n] = tune(n, X, y, s).best_estimator_
    return m

def train_score(name, model, X, y, s):
    """5-fold CV score on the training rows only -- decides which features a model keeps."""
    est = PAPER[name](s) if name in PAPER else model
    if name == 'RF (paper)': est.set_params(n_estimators=200)
    return cross_val_score(est, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=s)).mean()

if RUN_ANN:
    import tensorflow as tf
    from tensorflow.keras import Sequential, Input
    from tensorflow.keras.layers import Dense
    from tensorflow.keras.optimizers import Adam
    from tensorflow.keras.callbacks import EarlyStopping
    from tensorflow.keras.utils import to_categorical

    def _net(seed, n_in, lr):
        tf.keras.utils.set_random_seed(seed)
        m = Sequential([Input((n_in,)), Dense(74, activation='relu'), Dense(78, activation='relu'),
                        Dense(91, activation='relu'), Dense(3, activation='softmax')])
        m.compile(Adam(lr), 'categorical_crossentropy', metrics=['accuracy']); return m

    class TunedANN:
        def __init__(self, seed): self.seed = seed
        def tune(self, df):
            itr, iva = train_test_split(df.index, test_size=0.2, stratify=df.Fault, random_state=100 + self.seed); best = None
            for lr in [0.001, 0.0062]:
                for v in ['base', 'iscg']:
                    sc = StandardScaler().fit(feats(df.loc[itr], v)); m = _net(self.seed, sc.n_features_in_, lr)
                    h = m.fit(sc.transform(feats(df.loc[itr], v)), to_categorical(df.Fault[itr], 3), epochs=1000, batch_size=32, verbose=0,
                              validation_data=(sc.transform(feats(df.loc[iva], v)), to_categorical(df.Fault[iva], 3)),
                              callbacks=[EarlyStopping(monitor='val_loss', patience=100, restore_best_weights=True)])
                    va, ep = max(h.history['val_accuracy']), int(np.argmin(h.history['val_loss'])) + 1
                    if best is None or va > best[0]: best = (va, lr, v, ep)
            _, self.lr, self.variant, self.epochs = best; return self
        def fit(self, df, weights=None):
            self.sc = StandardScaler().fit(feats(df, self.variant)); self.m = _net(self.seed, self.sc.n_features_in_, self.lr)
            self.m.fit(self.sc.transform(feats(df, self.variant)), to_categorical(df.Fault, 3), epochs=self.epochs, batch_size=32,
                       verbose=0, sample_weight=weights); return self
        def predict(self, df):
            return self.m.predict(self.sc.transform(feats(df, self.variant)), verbose=0).argmax(1)

## A. System 1 — main result (realistic weather, I-V features, high-resolution sensor)

In [5]:
def system1_table(d, variants=('base', 'iscg', 'phys')):
    rows = []
    for run in range(RUNS):
        tr, te = split(d, run)
        for v in variants:
            Xtr, Xte, ytr = feats(d.loc[tr], v), feats(d.loc[te], v), d.Fault[tr]
            for n, m in all_models(Xtr, ytr, run).items():
                rows.append(dict(model=n, variant=v, run=run, train_cv=train_score(n, m, Xtr, ytr, run), **metrics(d.Fault[te], m.predict(Xte))))
        if RUN_ANN:
            a = TunedANN(run).tune(d.loc[tr]).fit(d.loc[tr])
            rows.append(dict(model='ANN (tuned)', variant='chosen per run', chosen=a.variant, run=run, **metrics(d.Fault[te], a.predict(d.loc[te]))))
        print('A run', run, 'done', flush=True)
    return pd.DataFrame(rows)

resA = system1_table(S1['realistic_hires_IV']); resA.to_csv('final_A_system1.csv', index=False)

def apply_rule(res):
    g = res.groupby(['model', 'variant']).agg(train_cv=('train_cv', 'mean'), Accuracy=('Accuracy', 'mean'), Precision=('Precision', 'mean'),
                                               Recall=('Recall', 'mean'), F1=('F1', 'mean'), std=('Accuracy', 'std')).reset_index()
    pick = g[g.train_cv.notna()].sort_values('train_cv').groupby('model').tail(1)
    return pd.concat([pick, g[g.train_cv.isna()]]).set_index('model').round(3)

display(resA.groupby(['model', 'variant']).Accuracy.mean().unstack().round(3))
PAPER_S1 = {'ANN (tuned)': 0.983, 'SVM (tuned)': 0.977, 'SVM (paper)': 0.977, 'RF (paper)': 0.957, 'DT (paper)': 0.930, 'LR (paper)': 0.787}
tabA = apply_rule(resA); tabA['paper (real data)'] = [PAPER_S1.get(m, np.nan) for m in tabA.index]
display(tabA)

A run 0 done
A run 1 done


A run 2 done
A run 3 done
A run 4 done


variant,base,chosen per run,iscg,phys
model,,,,
ANN (tuned),NaN,0.974,NaN,NaN
DT (paper),0.832,NaN,0.950,0.958
LR (paper),0.974,NaN,0.974,0.976
RF (paper),0.880,NaN,0.976,0.974
SVM (paper),0.956,NaN,0.966,0.968
SVM (tuned),0.972,NaN,0.972,0.972
XGBoost (tuned),0.884,NaN,0.972,0.970


,variant,train_cv,Accuracy,Precision,Recall,F1,std,paper (real data)
model,,,,,,,,
SVM (paper),phys,0.966,0.968,0.971,0.968,0.968,0.024,0.977
DT (paper),phys,0.967,0.958,0.959,0.958,0.958,0.024,0.930
XGBoost (tuned),iscg,0.977,0.972,0.973,0.972,0.972,0.011,NaN
LR (paper),phys,0.977,0.976,0.978,0.976,0.976,0.018,0.787
RF (paper),phys,0.978,0.974,0.976,0.974,0.974,0.015,0.957
SVM (tuned),phys,0.983,0.972,0.975,0.972,0.972,0.022,0.977
ANN (tuned),chosen per run,NaN,0.974,0.976,0.974,0.974,0.018,0.983


## C. System 2 — zero-shot and few-shot: train on simulated System 1, test on simulated System 2
Few-shot: add k simulated System 2 rows per class to training (weight 5), test on the rest. (Paper, real System 2: best SVM 0.627.)

In [6]:
src = S1['realistic_4f']; tgt = S2['realistic_4f']; rows = []
for v in ['base', 'iscg']:
    Xs, ys = feats(src, v), src.Fault; cv = StratifiedKFold(5, shuffle=True, random_state=0)
    svm_p = GridSearchCV(SVC(kernel='rbf'), {'gamma': np.logspace(-1, 3, 9), 'C': np.logspace(-1, 3, 9)}, cv=cv).fit(Xs, ys).best_params_
    xgb_p = RandomizedSearchCV(XGBClassifier(random_state=0, n_jobs=1), XGB_SPACE, n_iter=25, cv=cv, random_state=0).fit(Xs, ys).best_params_
    makers = {**{n: (lambda f=f: f(0)) for n, f in PAPER.items()},
              'SVM (tuned)': lambda: SVC(kernel='rbf', **svm_p), 'XGBoost (tuned)': lambda: XGBClassifier(random_state=0, n_jobs=1, **xgb_p)}
    for n, mk in makers.items():
        m = mk().fit(Xs, ys)
        rows.append(dict(model=n, variant=v, test='zero-shot: sim System 2', Accuracy=accuracy_score(tgt.Fault, m.predict(feats(tgt, v)))))
        for k in [5, 10]:
            for draw in range(5):
                shot = tgt.groupby('Fault', group_keys=False).sample(k, random_state=100 * k + draw); rest = tgt.drop(shot.index)
                X = pd.concat([Xs, feats(shot, v)]); y = pd.concat([ys, shot.Fault]); w = np.r_[np.ones(len(Xs)), np.full(len(shot), 5.0)]
                rows.append(dict(model=n, variant=v, test=f'few-shot k={k}: sim System 2', Accuracy=accuracy_score(rest.Fault, mk().fit(X, y, sample_weight=w).predict(feats(rest, v)))))
    print('C', v, 'done', flush=True)
if RUN_ANN:   # zero-shot over 5 seeds; few-shot re-uses the seed-0 settings (tuned on System 1 only)
    for seed in range(5):
        a = TunedANN(seed).tune(src).fit(src)
        rows.append(dict(model='ANN (tuned)', variant='chosen', test='zero-shot: sim System 2', Accuracy=accuracy_score(tgt.Fault, a.predict(tgt))))
        if seed == 0: a0 = a
    for k in [5, 10]:
        for draw in range(5):
            shot = tgt.groupby('Fault', group_keys=False).sample(k, random_state=100 * k + draw); rest = tgt.drop(shot.index)
            a = TunedANN(draw); a.lr, a.variant, a.epochs = a0.lr, a0.variant, a0.epochs
            a.fit(pd.concat([src, shot], ignore_index=True), np.r_[np.ones(len(src)), np.full(len(shot), 5.0)])
            rows.append(dict(model='ANN (tuned)', variant='chosen', test=f'few-shot k={k}: sim System 2', Accuracy=accuracy_score(rest.Fault, a.predict(rest))))
    print('C ANN done', flush=True)
resC = pd.DataFrame(rows); resC.to_csv('final_C_transfer.csv', index=False)
display(resC.groupby(['test', 'model', 'variant']).Accuracy.mean().unstack('variant').round(3))

C base done
C iscg done
C ANN done


variant                                       base  chosen   iscg
test                        model                                
few-shot k=10: sim System 2 ANN (tuned)        NaN   0.966    NaN
                            DT (paper)       0.914     NaN  0.944
                            LR (paper)       0.979     NaN  0.973
                            RF (paper)       0.976     NaN  0.994
                            SVM (paper)      0.976     NaN  0.993
                            SVM (tuned)      0.991     NaN  0.994
                            XGBoost (tuned)  0.954     NaN  0.996
few-shot k=5: sim System 2  ANN (tuned)        NaN   0.909    NaN
                            DT (paper)       0.785     NaN  0.945
                            LR (paper)       0.965     NaN  0.969
                            RF (paper)       0.936     NaN  0.973
                            SVM (paper)      0.962     NaN  0.965
                            SVM (tuned)      0.965     NaN  0.966
                            XGBoost (tuned)  0.907     NaN  0.995
zero-shot: sim System 2     ANN (tuned)        NaN   0.696    NaN
                            DT (paper)       0.695     NaN  0.775
                            LR (paper)       0.800     NaN  0.800
                            RF (paper)       0.785     NaN  0.805
                            SVM (paper)      0.660     NaN  0.895
                            SVM (tuned)      0.805     NaN  0.800
                            XGBoost (tuned)  0.800     NaN  0.805

## D. Side result: System 2 trained and tested on simulated System 2 (same protocol as System 1)

In [7]:
rows = []
for name, d in S2.items():
    for run in range(RUNS):
        tr, te = split(d, run)
        for v in ['base', 'iscg']:
            Xtr, ytr = feats(d.loc[tr], v), d.Fault[tr]
            for n, m in all_models(Xtr, ytr, run).items():
                rows.append(dict(data=name, model=n, variant=v, run=run, train_cv=train_score(n, m, Xtr, ytr, run),
                                 **metrics(d.Fault[te], m.predict(feats(d.loc[te], v)))))
        if RUN_ANN:
            a = TunedANN(run).tune(d.loc[tr]).fit(d.loc[tr])
            rows.append(dict(data=name, model='ANN (tuned)', variant='chosen', run=run, **metrics(d.Fault[te], a.predict(d.loc[te]))))
    print('D', name, 'done', flush=True)
resD = pd.DataFrame(rows); resD.to_csv('final_D_system2_within.csv', index=False)
display(resD.groupby(['data', 'model', 'variant']).Accuracy.mean().unstack('variant').round(3))

D realistic_4f done
D realistic_IV done


variant                        base  chosen   iscg
data         model                                
realistic_4f ANN (tuned)        NaN   1.000    NaN
             DT (paper)       0.910     NaN  0.995
             LR (paper)       0.995     NaN  1.000
             RF (paper)       0.955     NaN  0.995
             SVM (paper)      0.985     NaN  1.000
             SVM (tuned)      0.990     NaN  0.990
             XGBoost (tuned)  0.935     NaN  0.995
realistic_IV ANN (tuned)        NaN   0.975    NaN
             DT (paper)       0.975     NaN  0.980
             LR (paper)       0.995     NaN  0.995
             RF (paper)       0.990     NaN  0.990
             SVM (paper)      0.985     NaN  0.980
             SVM (tuned)      0.985     NaN  0.985
             XGBoost (tuned)  0.980     NaN  0.980

## F. Paper-style outputs (System 1 final models)
P/R/F1, confusion matrix of the run closest to the mean, randomized-block ANOVA (model × run; needs RUNS ≥ 2),
and the measurement-error sensitivity test of the paper's Fig. 11 (uniform ±e error on test voltages and currents).

In [8]:
from scipy import stats
d = S1['realistic_hires_IV']
best_v = resA[resA.train_cv.notna()].groupby(['model', 'variant']).train_cv.mean().reset_index().sort_values('train_cv').groupby('model').tail(1).set_index('model').variant.to_dict()
FINAL = ['SVM (tuned)', 'XGBoost (tuned)', 'RF (paper)', 'DT (paper)', 'LR (paper)']
def perturb(df, e, rng):
    x = df.copy()
    for c in ['Voc/MaxVoc', 'Isc/MaxIsc', 'Vmp/MaxVoc', 'Imp/MaxIsc']:
        x[c] = x[c] * (1 + rng.uniform(-e, e, len(x)))
    x['FF'] = x['Vmp/MaxVoc'] * x['Imp/MaxIsc'] / (x['Voc/MaxVoc'] * x['Isc/MaxIsc'])
    return x
def fit_final(n, X, y, s):
    return PAPER[n](s).fit(X, y) if n in PAPER else tune(n, X, y, s).best_estimator_
rowsF, cms, sens = [], {}, []
for run in range(RUNS):
    tr, te = split(d, run); rng = np.random.default_rng(run)
    fitted = {n: fit_final(n, feats(d.loc[tr], best_v[n]), d.Fault[tr], run) for n in FINAL}
    if RUN_ANN: ann = TunedANN(run).tune(d.loc[tr]).fit(d.loc[tr])
    preds = {n: m.predict(feats(d.loc[te], best_v[n])) for n, m in fitted.items()}
    if RUN_ANN: preds['ANN (tuned)'] = ann.predict(d.loc[te])
    for n, p in preds.items():
        rowsF.append(dict(model=n, run=run, **metrics(d.Fault[te], p))); cms[(n, run)] = confusion_matrix(d.Fault[te], p, labels=[0, 1, 2])
    for e in [0.0, 0.01, 0.05, 0.10, 0.15, 0.20]:
        for rep in range(3):
            noisy = perturb(d.loc[te], e, rng)
            for n, m in fitted.items():
                sens.append(dict(model=n, error=e, Accuracy=accuracy_score(d.Fault[te], m.predict(feats(noisy, best_v[n])))))
            if RUN_ANN: sens.append(dict(model='ANN (tuned)', error=e, Accuracy=accuracy_score(d.Fault[te], ann.predict(noisy))))
    print('F run', run, 'done', flush=True)
RF_ = pd.DataFrame(rowsF)
display(RF_.groupby('model')[['Accuracy', 'Precision', 'Recall', 'F1']].mean().round(4))
for n, g in RF_.groupby('model'):
    r = g.iloc[(g.Accuracy - g.Accuracy.mean()).abs().argmin()].run
    print(n, '- confusion matrix, run', r, '(rows = true normal / shading / dirt)'); print(cms[(n, r)])
if RUNS >= 2:
    an = []
    for metric in ['Accuracy', 'Precision', 'Recall', 'F1']:
        M = RF_.pivot(index='model', columns='run', values=metric).values; k, b = M.shape; g = M.mean()
        ssm = b * ((M.mean(1) - g) ** 2).sum(); ssb = k * ((M.mean(0) - g) ** 2).sum(); sse = ((M - g) ** 2).sum() - ssm - ssb
        dfm, dfe = k - 1, (k - 1) * (b - 1); F = (ssm / dfm) / (sse / dfe)
        an.append(dict(metric=metric, SS=ssm, DF=dfm, F=F, p=stats.f.sf(F, dfm, dfe), np2=ssm / (ssm + sse)))
    display(pd.DataFrame(an).round(5))
sensT = pd.DataFrame(sens).groupby(['model', 'error']).Accuracy.mean().unstack()
sensT.loc['Paper ANN (Fig. 11)'] = [0.9833, 0.9333, 0.6667, 0.5333, 0.45, 0.4167]
display(sensT.round(3))

F run 0 done
F run 1 done
F run 2 done
F run 3 done
F run 4 done


,Accuracy,Precision,Recall,F1
model,,,,
ANN (tuned),0.974,0.9756,0.974,0.9739
DT (paper),0.958,0.9590,0.958,0.9578
LR (paper),0.976,0.9782,0.976,0.9759
RF (paper),0.974,0.9758,0.974,0.9739
SVM (tuned),0.972,0.9748,0.972,0.9718
XGBoost (tuned),0.972,0.9731,0.972,0.9720


ANN (tuned) - confusion matrix, run 1 (rows = true normal / shading / dirt)
[[31  0  2]
 [ 0 34  0]
 [ 1  0 32]]
DT (paper) - confusion matrix, run 0 (rows = true normal / shading / dirt)
[[33  0  1]
 [ 0 33  0]
 [ 2  1 30]]
LR (paper) - confusion matrix, run 0 (rows = true normal / shading / dirt)
[[34  0  0]
 [ 0 33  0]
 [ 2  0 31]]
RF (paper) - confusion matrix, run 0 (rows = true normal / shading / dirt)
[[33  0  1]
 [ 0 33  0]
 [ 2  0 31]]
SVM (tuned) - confusion matrix, run 0 (rows = true normal / shading / dirt)
[[34  0  0]
 [ 0 33  0]
 [ 2  1 30]]
XGBoost (tuned) - confusion matrix, run 0 (rows = true normal / shading / dirt)
[[33  0  1]
 [ 0 33  0]
 [ 2  0 31]]


,metric,SS,DF,F,p,np2
0,Accuracy,0.00107,5,3.17822,0.02848,0.44276
1,Precision,0.00121,5,3.72424,0.01518,0.48215
2,Recall,0.00107,5,3.17822,0.02848,0.44276
3,F1,0.00108,5,3.16364,0.02897,0.44162


error,0.00,0.01,0.05,0.10,0.15,0.20
model,,,,,,
ANN (tuned),0.974,0.926,0.724,0.612,0.555,0.518
DT (paper),0.958,0.934,0.658,0.565,0.517,0.510
LR (paper),0.976,0.971,0.831,0.687,0.628,0.584
RF (paper),0.974,0.964,0.671,0.571,0.527,0.518
SVM (tuned),0.972,0.968,0.867,0.719,0.643,0.583
XGBoost (tuned),0.972,0.911,0.619,0.524,0.505,0.499
Paper ANN (Fig. 11),0.983,0.933,0.667,0.533,0.450,0.417


## G. Robustness fix: noise-augmented training
Training rows plus copies with ±1 % and ±3 % voltage/current error; same test rows and sensitivity test.

In [9]:
na = []
for run in range(RUNS):
    tr, te = split(d, run); rng = np.random.default_rng(1000 + run)
    aug = pd.concat([d.loc[tr]] + [perturb(d.loc[tr], e, rng) for e in (0.01, 0.01, 0.03, 0.03)], ignore_index=True)
    rng2 = np.random.default_rng(run)
    for n in ['XGBoost (tuned)', 'SVM (tuned)', 'RF (paper)', 'LR (paper)']:
        m = fit_final(n, feats(aug, best_v[n]), aug.Fault, run)
        for e in [0.0, 0.01, 0.05, 0.10]:
            for rep in range(3):
                na.append(dict(model=n, error=e, Accuracy=accuracy_score(d.Fault[te], m.predict(feats(perturb(d.loc[te], e, rng2), best_v[n])))))
    print('G run', run, 'done', flush=True)
display(pd.DataFrame(na).groupby(['model', 'error']).Accuracy.mean().unstack().round(3))

G run 0 done
G run 1 done
G run 2 done
G run 3 done
G run 4 done


error,0.00,0.01,0.05,0.10
model,,,,
LR (paper),0.968,0.966,0.883,0.784
RF (paper),0.966,0.968,0.879,0.715
SVM (tuned),0.932,0.937,0.785,0.427
XGBoost (tuned),0.954,0.949,0.844,0.660


## E. Summary (System 2 columns show both feature variants — choosing the better one would mean choosing by test results)
Paper reference: System 1 (real) ANN 0.983, SVM 0.977; System 2 (real, zero-shot) best SVM 0.627.

In [10]:
zs = resC[resC.test == 'zero-shot: sim System 2'].groupby(['model', 'variant']).Accuracy.mean().unstack()
fs = resC[resC.test == 'few-shot k=5: sim System 2'].groupby(['model', 'variant']).Accuracy.mean().unstack()
summary = pd.DataFrame({'S1 sim (realistic, I-V, hi-res)': tabA['Accuracy'], 'S1 paper (real)': tabA['paper (real data)'],
                        'S2 sim zero-shot': zs.get('base'), 'S2 sim zero-shot +Isc/G': zs.get('iscg'),
                        'S2 sim few-shot k=5': fs.get('base'), 'S2 sim few-shot k=5 +Isc/G': fs.get('iscg')}).round(3)
if 'chosen' in zs.columns:
    summary.loc['ANN (tuned)', ['S2 sim zero-shot', 'S2 sim few-shot k=5']] = [zs.loc['ANN (tuned)', 'chosen'], fs.loc['ANN (tuned)', 'chosen']]
summary.to_csv('final_SUMMARY.csv'); display(summary)
print('NOTEBOOK COMPLETE')

,"S1 sim (realistic, I-V, hi-res)",S1 paper (real),S2 sim zero-shot,S2 sim zero-shot +Isc/G,S2 sim few-shot k=5,S2 sim few-shot k=5 +Isc/G
model,,,,,,
ANN (tuned),0.974,0.983,0.696,NaN,0.909189,NaN
DT (paper),0.958,0.930,0.695,0.775,0.785000,0.945
LR (paper),0.976,0.787,0.800,0.800,0.965000,0.969
RF (paper),0.974,0.957,0.785,0.805,0.936000,0.973
SVM (paper),0.968,0.977,0.660,0.895,0.962000,0.965
SVM (tuned),0.972,0.977,0.805,0.800,0.965000,0.966
XGBoost (tuned),0.972,NaN,0.800,0.805,0.907000,0.995


NOTEBOOK COMPLETE
